<a href="https://colab.research.google.com/github/justkousei/Planeacion-Agregada/blob/main/planeacion_agregada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<br>

<p align="center" style="text-align:center"><b>UNIVERSIDAD DISTRITAL FRANCISCO JOSÉ DE CALDAS</b><br>Investigación de Operaciones I</p>

<hr>

<h1 align="center" style="text-align:center">Planeación Agregada Multietapa, Multifuente y Multiproducto</h1>

<p align="center" style="text-align:center">Modelo de programación lineal entera mixta (MILP) con escasez<br>Implementación en Python y Google Colab</p>

<hr>

<p align="center" style="text-align:center"><b>Integrantes</b><br>
Juan Felipe Gallego Chaves · 20251020016<br>
Julian David Muñoz Revelo · 20251020042<br>
Juan Esteban Guerrero Nieto · 20251020053</p>

<p align="center" style="text-align:center"><b>Docente</b><br>Luis Alfonso Peña Florez</p>

<p align="center" style="text-align:center">Bogotá D.C., noviembre de 2026</p>

<hr>

## Cómo usar este cuaderno

Este cuaderno calcula **cuánto producir, en qué máquina y en qué período** para cumplir la demanda al menor costo total, en un sistema de varias etapas y con uno o varios productos. Todos los datos se pueden cambiar desde una interfaz, sin tocar el código.

**Paso 0 · Ejecutar**
- Pulsar el botón de reproducir (el círculo con el triángulo, a la izquierda de la celda) de código que está debajo, o ir al menú **Entorno de ejecución → Ejecutar todo**.
- Esperar unos segundos. La interfaz aparece justo debajo de la celda.

**Paso 1 · Definir la estructura**
- **Productos:** nombres separados por coma. Ejemplo: `Producto A, Producto B`.
- **Fuentes por etapa:** un número por etapa, separados por coma. Ejemplo: `3,2,3,4` significa 4 etapas con 3, 2, 3 y 4 máquinas alternativas.
- **Nº de períodos:** cuántos períodos tiene el plan. Ejemplo: `4`.
- **Meses por período:** duración de cada período. Ejemplo: `3` (un trimestre).
- **Factor de eficiencia:** fracción del tiempo en que la máquina produce. Ejemplo: `0,8`.
- **Escasez:** marcar la casilla si se permite entregar la demanda con retraso.
- **Resolver:** dejar **Ambas (comparar)**. Tiempo máximo y gap se pueden dejar como están.

**Paso 2 · Generar las tablas**
- Pulsar el botón azul **Paso 2 · Generar tablas**.
- Si solo se quiere probar, pulsar **Cargar caso del artículo**: llena todo con el ejemplo del paper y se puede ir directo al Paso 4.

**Paso 3 · Editar los datos**
- Aparecen 5 pestañas: **1 · Demanda**, **2 · Capacidad**, **3 · Velocidad y setup**, **4 · Costos** y **5 · Inventarios**.
- En la pestaña **5 · Inventarios** podrás configurar el **inventario inicial para cada una de las etapas** (Ej. Etapa 1: 100, Etapa 2: 50, Etapa 3: 150) además de los costos de mantener.

**Paso 4 · Resolver y revisar**
- Pulsar el botón verde **Paso 4 · RESOLVER**.
- El solver tomará en cuenta los inventarios iniciales de cada etapa para optimizar la producción.

In [5]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  PARTE 1 · MOTOR MATEMÁTICO (modelo MILP del artículo, generalizado) ║
# ╚══════════════════════════════════════════════════════════════════════╝
import time
import numpy as np
import pandas as pd
from dataclasses import dataclass
from scipy.optimize import milp, linprog, LinearConstraint, Bounds
from scipy.sparse import coo_matrix


@dataclass
class Datos:
    productos: list      # nombres de productos
    J: list              # J[i-1] = nº de fuentes de la etapa i   (M = len(J))
    T: int               # nº de períodos
    meses: float         # meses que dura cada período (para pérdidas por mantenimiento)
    efic: float          # factor de eficiencia (0.8 en el artículo)
    dias: list           # días hábiles por período (len T)
    fuente: dict         # (i,j) -> dict(N, perd, turnos, horas)
    tasa: dict           # (p,i,j) -> unidades/hora   (0 = esa fuente no procesa ese producto)
    setup: dict          # (p,i,j) -> costo de preparación
    costo: dict          # (p,i,j,t) -> costo unitario de producción
    mantener: dict       # (p,i) -> costo de mantener inventario por unidad y período
    coef: dict           # (p,i) -> unidades de la etapa i necesarias por unidad de la etapa i+1
    demanda: dict        # (p,t) -> demanda
    inv_ini: list        # inventario inicial de producto terminado, por producto
    inv_fin: list        # inventario final de producto terminado, por producto
    c_falt: list         # costo de escasez (unidad pendiente) por producto
    inv_ini_etapas: dict = None  # (p, i) -> inventario inicial de la etapa i (i < M). Por defecto 0
    inv_fin_etapas: dict = None  # (p, i) -> inventario final requerido de la etapa i (i < M). Por defecto 0
    faltantes: bool = True    # ¿se permite demanda pendiente (escasez)?
    enteras: bool = True      # ¿X enteras?
    t_limite: float = 60.0    # segundos máx. del solver
    gap: float = 0.0          # gap relativo tolerado (%)


def horas(d, i, j, t):
    """Horas-máquina disponibles de la fuente (i,j) en el período t.
       Cd = N · tasa · efic · ( días·turnos·h_turno − pérdida_mant·meses )   (Tabla III del artículo)"""
    f = d.fuente[(i, j)]
    h = f['N'] * d.efic * (d.dias[t - 1] * f['turnos'] * f['horas'] - f['perd'] * d.meses)
    return max(h, 0.0)


class _Modelo:
    def __init__(s):
        s.nom, s.lb, s.ub, s.ent, s.c = [], [], [], [], []
        s.r, s.k, s.v, s.lo, s.hi = [], [], [], [], []

    def var(s, nom, lb, ub, ent, c):
        s.nom.append(nom); s.lb.append(lb); s.ub.append(ub)
        s.ent.append(1 if ent else 0); s.c.append(c)
        return len(s.nom) - 1

    def con(s, terms, lo, hi):
        r = len(s.lo)
        for k, v in terms:
            s.r.append(r); s.k.append(k); s.v.append(v)
        s.lo.append(lo); s.hi.append(hi)

    def A(s):
        return coo_matrix((s.v, (s.r, s.k)), shape=(len(s.lo), len(s.nom))).tocsr()


ESTADOS = {0: "Óptima", 1: "Límite de tiempo (mejor solución encontrada)",
           2: "INFACTIBLE", 3: "No acotado", 4: "Error numérico"}


def resolver(d: Datos):
    P, M, T = len(d.productos), len(d.J), d.T
    sep = "" if max(M, max(d.J), T) < 10 else "_"
    m = _Modelo()
    X, Y, I, Ip, Im, CAP = {}, {}, {}, {}, {}, {}

    inv_ini_et = d.inv_ini_etapas if d.inv_ini_etapas is not None else {}
    inv_fin_et = d.inv_fin_etapas if d.inv_fin_etapas is not None else {}

    def nm(letra, p, *ix):
        s = letra + sep.join(str(a) for a in ix)
        return s + (f" [{d.productos[p]}]" if P > 1 else "")

    # ---------------- variables
    for p in range(P):
        for i in range(1, M + 1):
            for t in range(1, T + 1):
                for j in range(1, d.J[i - 1] + 1):
                    cap = horas(d, i, j, t) * d.tasa[(p, i, j)]
                    if d.enteras: cap = float(np.floor(cap + 1e-9))
                    CAP[p, i, j, t] = cap
                    X[p, i, j, t] = m.var(nm("X", p, i, j, t), 0, cap, d.enteras, d.costo[(p, i, j, t)])
    for p in range(P):
        for i in range(1, M):
            for t in range(1, T + 1):
                if t < T:
                    I[p, i, t] = m.var(nm("I", p, i, t), 0, np.inf, False, d.mantener[(p, i)])
                else:
                    # Forzar inventario final requerido para etapa intermedia
                    req_fin = inv_fin_et.get((p, i), 0.0)
                    I[p, i, t] = m.var(nm("I", p, i, t), req_fin, req_fin, False, d.mantener[(p, i)])
        for t in range(1, T + 1):
            if t < T: Ip[p, t] = m.var(nm("I+", p, M, t), 0, np.inf, False, d.mantener[(p, M)])
            else:     Ip[p, t] = m.var(nm("I+", p, M, t), d.inv_fin[p], d.inv_fin[p], False, d.mantener[(p, M)])
        for t in range(1, T + 1):
            ub = np.inf if (d.faltantes and t < T) else 0
            Im[p, t] = m.var(nm("I-", p, M, t), 0, ub, False, d.c_falt[p])
    for p in range(P):
        for i in range(1, M + 1):
            for t in range(1, T + 1):
                for j in range(1, d.J[i - 1] + 1):
                    Y[p, i, j, t] = m.var(nm("Y", p, i, j, t), 0, 1 if CAP[p, i, j, t] > 0 else 0,
                                          True, d.setup[(p, i, j)])

    # ---------------- restricciones
    for p in range(P):
        for t in range(1, T + 1):
            for i in range(1, M):                      # balance de las etapas 1..M-1
                terms = [(X[p, i, j, t], 1) for j in range(1, d.J[i - 1] + 1)]
                rhs = 0.0
                if t > 1:
                    terms.append((I[p, i, t - 1], 1))
                else:
                    rhs -= inv_ini_et.get((p, i), 0.0)
                terms.append((I[p, i, t], -1))
                terms += [(X[p, i + 1, j, t], -d.coef[(p, i)]) for j in range(1, d.J[i] + 1)]
                m.con(terms, rhs, rhs)

            terms = [(X[p, M, j, t], 1) for j in range(1, d.J[M - 1] + 1)]   # balance etapa final
            rhs = d.demanda[(p, t)]
            if t > 1: terms += [(Ip[p, t - 1], 1), (Im[p, t - 1], -1)]
            else:     rhs -= d.inv_ini[p]
            terms += [(Ip[p, t], -1), (Im[p, t], 1)]
            m.con(terms, rhs, rhs)
    for (p, i, j, t), cap in CAP.items():              # X ≤ Cd · Y
        if cap > 0:
            m.con([(X[p, i, j, t], 1), (Y[p, i, j, t], -cap)], -np.inf, 0)
    if P > 1:                                           # horas-máquina compartidas entre productos
        for i in range(1, M + 1):
            for j in range(1, d.J[i - 1] + 1):
                for t in range(1, T + 1):
                    terms = [(X[p, i, j, t], 1.0 / d.tasa[(p, i, j)]) for p in range(P) if d.tasa[(p, i, j)] > 0]
                    if terms: m.con(terms, -np.inf, horas(d, i, j, t))

    # ---------------- resolver MILP
    c, A = np.array(m.c, float), m.A()
    lo, hi = np.array(m.lo, float), np.array(m.hi, float)
    lb, ub = np.array(m.lb, float), np.array(m.ub, float)
    t0 = time.time()
    res = milp(c, constraints=LinearConstraint(A, lo, hi), integrality=np.array(m.ent),
               bounds=Bounds(lb, ub),
               options={"time_limit": float(d.t_limite), "mip_rel_gap": d.gap / 100.0, "disp": False})
    seg = time.time() - t0
    if res.x is None:
        msg = ESTADOS.get(res.status, res.message)
        if res.status == 2:
            msg += (" → no hay forma de cumplir la demanda con las capacidades dadas. "
                    "Subir máquinas, turnos o días, bajar la demanda, o activar 'permitir escasez'.")
        return {"ok": False, "estado": msg, "seg": seg}

    x = res.x.copy()
    for k in range(len(x)):
        if m.ent[k]: x[k] = round(x[k])

    # ---------------- costos reducidos
    rc, xlp = np.full(len(x), np.nan), x
    try:
        lb2, ub2 = lb.copy(), ub.copy()
        for k in Y.values(): lb2[k] = ub2[k] = x[k]
        eq = lo == hi
        r2 = linprog(c, A_ub=A[~eq] if (~eq).any() else None, b_ub=hi[~eq] if (~eq).any() else None,
                     A_eq=A[eq] if eq.any() else None, b_eq=lo[eq] if eq.any() else None,
                     bounds=np.column_stack([lb2, ub2]), method="highs")
        if r2.status == 0:
            rc = r2.lower.marginals + r2.upper.marginals
            xlp = r2.x
    except Exception:
        pass

    # ---------------- tabla solución
    yset, filas = set(Y.values()), []
    for k in range(len(x)):
        if k in yset:
            est, rcv = "Fija (binaria)", np.nan
        else:
            tol = 1e-6 * (1 + abs(xlp[k]))
            enc = abs(xlp[k] - lb[k]) < tol or abs(xlp[k] - ub[k]) < tol
            est, rcv = ("En cota" if enc else "Básica"), rc[k]
        filas.append([m.nom[k], x[k], m.c[k], x[k] * m.c[k], rcv, est])
    tabla = pd.DataFrame(filas, columns=["Variable", "Valor solución", "Costo unitario",
                                         "Contribución total", "Costo reducido", "Estado base"])

    # ---------------- resúmenes
    cost = dict(
        prod=sum(m.c[k] * x[k] for k in X.values()),
        setup=sum(m.c[k] * x[k] for k in Y.values()),
        mant=sum(m.c[k] * x[k] for k in list(I.values()) + list(Ip.values())),
        falt=sum(m.c[k] * x[k] for k in Im.values()))
    cols = [f"T{t}" for t in range(1, T + 1)]
    plan, inv = {}, {}
    for p, nombre in enumerate(d.productos):
        idx, vals = [], []
        for i in range(1, M + 1):
            for j in range(1, d.J[i - 1] + 1):
                idx.append(f"Etapa {i} · Fuente {j}")
                vals.append([x[X[p, i, j, t]] for t in range(1, T + 1)])
        plan[nombre] = pd.DataFrame(vals, index=idx, columns=cols)
        idx, vals = [], []
        idx.append("Demanda"); vals.append([d.demanda[(p, t)] for t in range(1, T + 1)])
        idx.append(f"Producción etapa {M}")
        vals.append([sum(x[X[p, M, j, t]] for j in range(1, d.J[M - 1] + 1)) for t in range(1, T + 1)])
        for i in range(1, M):
            idx.append(f"Inventario etapa {i}"); vals.append([x[I[p, i, t]] for t in range(1, T + 1)])
        idx.append(f"Inv. prod. terminado (I+)"); vals.append([x[Ip[p, t]] for t in range(1, T + 1)])
        idx.append(f"Demanda pendiente (I-)");    vals.append([x[Im[p, t]] for t in range(1, T + 1)])
        inv[nombre] = pd.DataFrame(vals, index=idx, columns=cols)
    util = {}
    for i in range(1, M + 1):
        for j in range(1, d.J[i - 1] + 1):
            fila = []
            for t in range(1, T + 1):
                H = horas(d, i, j, t)
                usado = sum(x[X[p, i, j, t]] / d.tasa[(p, i, j)] for p in range(P) if d.tasa[(p, i, j)] > 0)
                fila.append(usado / H if H > 0 else 0.0)
            util[f"E{i}-F{j}"] = fila
    util = pd.DataFrame(util, index=cols).T

    return {"ok": True, "estado": ESTADOS.get(res.status, str(res.status)), "obj": float(res.fun),
            "seg": seg, "tabla": tabla, "costos": cost, "plan": plan, "inv": inv, "util": util,
            "gap": getattr(res, "mip_gap", None)}


# ───────────── caso del artículo (Tablas I–III) ─────────────
def caso_articulo():
    J = [3, 2, 3, 4]
    fte = {  # (i,j): (N máquinas, pérdida h/mes/máq)
        (1, 1): (5, 12.2), (1, 2): (5, 10.3), (1, 3): (10, 11.2),
        (2, 1): (12, 14.2), (2, 2): (11, 16.1),
        (3, 1): (4, 12.2), (3, 2): (5, 12.5), (3, 3): (11, 13.5),
        (4, 1): (3, 12.5), (4, 2): (4, 11.2), (4, 3): (5, 10.3), (4, 4): (6, 11.4)}
    rate = {(1, 1): 20, (1, 2): 25, (1, 3): 28, (2, 1): 35, (2, 2): 40, (3, 1): 25, (3, 2): 28, (3, 3): 30,
            (4, 1): 24, (4, 2): 25, (4, 3): 28, (4, 4): 30}
    setup = {(1, 1): 25000, (1, 2): 24000, (1, 3): 20000, (2, 1): 24000, (2, 2): 22000,
             (3, 1): 12000, (3, 2): 10000, (3, 3): 12000,
             (4, 1): 23000, (4, 2): 25000, (4, 3): 23000, (4, 4): 22000}
    cu = {(1, 1): [25, 28, 30, 28], (1, 2): [28, 30, 30, 32], (1, 3): [30, 32, 34, 32],
          (2, 1): [45, 46, 44, 48], (2, 2): [48, 50, 49, 50],
          (3, 1): [12, 15, 18, 19], (3, 2): [14, 16, 18, 20], (3, 3): [13, 16, 20, 22],
          (4, 1): [24, 29, 24, 28], (4, 2): [24, 29, 24, 27], (4, 3): [26, 30, 27, 25], (4, 4): [28, 33, 27, 25]}
    return dict(J=J, T=4, meses=3, efic=0.8, dias=[67, 68, 76, 67], fte=fte, rate=rate, setup=setup, cu=cu,
                mantener=[2, 3, 4, 5], demanda=[50500, 65500, 82000, 92000],
                inv_ini=1000, inv_fin=1000, c_falt=4)


def datos_articulo():
    a = caso_articulo(); T = a['T']
    return Datos(
        productos=["Producto"], J=a['J'], T=T, meses=a['meses'], efic=a['efic'], dias=a['dias'],
        fuente={k: dict(N=v[0], perd=v[1], turnos=1, horas=8) for k, v in a['fte'].items()},
        tasa={(0,) + k: v for k, v in a['rate'].items()},
        setup={(0,) + k: v for k, v in a['setup'].items()},
        costo={(0,) + k + (t + 1,): v[t] for k, v in a['cu'].items() for t in range(T)},
        mantener={(0, i + 1): a['mantener'][i] for i in range(4)},
        coef={(0, i): 1.0 for i in range(1, 5)},
        demanda={(0, t + 1): a['demanda'][t] for t in range(T)},
        inv_ini=[a['inv_ini']], inv_fin=[a['inv_fin']], c_falt=[a['c_falt']], inv_ini_etapas={},
        inv_fin_etapas={})


In [7]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  PARTE 2 · INTERFAZ GRÁFICA (ipywidgets)                             ║
# ╚══════════════════════════════════════════════════════════════════════╝
import ipywidgets as W
from dataclasses import replace
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import numpy as np
import pandas as pd

S = {"est": None, "g": {}, "R": None, "d": None}

def ES(x, dec=2):
    if x is None or (isinstance(x, float) and np.isnan(x)): return "—"
    return f"{x:,.{dec}f}".replace(",", "§").replace(".", ",").replace("§", ".")

PAL = dict(total="#27ae60", prod="#2980b9", setup="#e67e22", mant="#8e44ad", falt="#e74c3c",
           info="#16a085", gris="#7f8c8d")
TIPO_COLOR = {"X": PAL["prod"], "I": PAL["mant"], "Y": PAL["setup"]}

def rgba(hexc, a):
    h = hexc.lstrip("#")
    return f"rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},{a})"

def seccion(titulo, color, cuerpo, icono=""):
    return (f"<div style='border:1px solid {rgba(color,.55)};border-radius:12px;margin:12px 0;overflow:hidden'>"
            f"<div style='background:{rgba(color,.9)};color:white;padding:8px 14px;font-weight:700;font-size:15px'>"
            f"{(icono + ' ' + titulo).strip()}</div><div style='padding:10px 14px'>{cuerpo}</div></div>")

CSS_TABLA = ("<style>.pa{border-collapse:collapse;width:100%}.pa td,.pa th{padding:4px 12px;"
             "border-bottom:1px solid #8883;text-align:right;white-space:nowrap}"
             ".pa th{position:sticky;top:0;color:white;z-index:1}"
             ".pa td:first-child,.pa th:first-child{text-align:left}"
             ".pa tr:hover td{background:#8882}</style>")

def tabla_html(df, alto=450, dec=2, color=None, calor=False, fila_color=None):
    color = color or PAL["prod"]
    num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    vmax = max([float(np.nanmax(np.abs(df[num].values))) if len(num) else 1.0, 1e-9])
    h = "<table class='pa'><thead><tr>" + "".join(
        f"<th style='background:{color}'>{c}</th>" for c in df.columns) + "</tr></thead><tbody>"
    for _, fila in df.iterrows():
        borde = f"border-left:5px solid {fila_color(fila)};" if fila_color else ""
        h += "<tr>"
        for n, c in enumerate(df.columns):
            v = fila[c]; est = borde if n == 0 else ""
            if c in num:
                txt = ES(v, dec)
                if calor and v == v and abs(v) > 1e-9: est += f"background:{rgba(color, .12 + .55*min(abs(v)/vmax,1))};"
                elif abs(v) < 1e-9 if v == v else False: est += "opacity:.4;"
            else: txt = str(v)
            h += f"<td style='{est}'>{txt}</td>"
        h += "</tr>"
    return f"{CSS_TABLA}<div style='max-height:{alto}px;overflow:auto'>{h}</tbody></table></div>"

COL_T = {"dem": "#2980b9", "cap": "#16a085", "tasa": "#e67e22", "costo": "#8e44ad", "etp": "#d4ac0d"}
CSS_UI = "<style>" + "".join(
    f".pa-{k} input{{border:2px solid {c}!important;background:{rgba(c, .22)}!important;"
    f"font-weight:700!important;text-align:right!important}}" for k, c in COL_T.items()) + (
    ".pa-config input,.pa-config select{border:2px solid #7f8c8d!important;font-weight:600!important}"
    "</style>")

def ayuda(titulo, color, que_es, campos=(), consejo=""):
    li = "".join(f"<li style='margin:3px 0'><b>{n}:</b> {d}</li>" for n, d in campos)
    lista = f"<ul style='margin:6px 0 6px 18px;padding:0'>{li}</ul>" if li else ""
    tip = (f"<div style='margin-top:6px'><i>{consejo}</i></div>" if consejo else "")
    return W.HTML(f"<div style='border-left:6px solid {color};background:{rgba(color, .13)};padding:10px 16px;"
                  f"border-radius:8px;margin:6px 0 12px 0;max-width:1000px'>"
                  f"<div style='font-weight:800;font-size:15px;color:{color}'>{titulo}</div>"
                  f"<div style='margin-top:4px'>{que_es}</div>{lista}{tip}</div>")

class Grid:
    def __init__(self, filas, cols, defaults=None, clave="dem", unidades=None, ancho=100, ancho_et=290):
        color = COL_T[clave]
        defaults = defaults if defaults is not None else [0.0] * len(cols)
        unidades = unidades or [""] * len(cols)
        items = [W.HTML("")]
        for c, u in zip(cols, unidades):
            u_html = f"<div style='font-weight:400;font-size:11px'>{u}</div>" if u else ""
            items.append(W.HTML(f"<div style='background:{color};color:white;padding:5px 6px;border-radius:8px;"
                                f"text-align:center;font-weight:700;font-size:12px;line-height:1.25'>{c}{u_html}</div>"))
        self.celdas = []
        for r, f in enumerate(filas):
            items.append(W.HTML(f"<div style='border-left:6px solid {color};background:{rgba(color, .10 if r % 2 == 0 else .20)};"
                                f"padding:7px 10px;border-radius:6px;font-weight:600'>{f}</div>"))
            fila = []
            for c in range(len(cols)):
                w = W.FloatText(value=float(defaults[c]), layout=W.Layout(width=f"{ancho}px"))
                w.add_class(f"pa-{clave}"); fila.append(w)
            items += fila; self.celdas.append(fila)
        self.box = W.GridBox(items, layout=W.Layout(
            grid_template_columns=f"{ancho_et}px repeat({len(cols)}, {ancho + 12}px)", grid_gap="6px 8px"))

    def get(self):
        return np.array([[c.value for c in fila] for fila in self.celdas], float)

    def set(self, M):
        for r, fila in enumerate(M):
            for c, v in enumerate(fila): self.celdas[r][c].value = float(v)

def _ancho(w, n=270):
    w.layout = W.Layout(width=f"{n}px"); w.add_class("pa-config"); return w

w_prod = _ancho(W.Text(value="Producto A, Producto B"))
w_J = _ancho(W.Text(value="3,2,3,4"))
w_T = _ancho(W.IntText(value=4))
w_meses = _ancho(W.FloatText(value=3))
w_efic = _ancho(W.FloatText(value=0.8))
w_falt = W.Checkbox(value=True, description="Sí, permitir escasez", indent=False)
w_modo = _ancho(W.Dropdown(options=["Ambas (comparar)", "Entera (con aproximación)", "Continua (sin aproximación)"],
                           value="Ambas (comparar)"))
w_tl = _ancho(W.IntText(value=60))
w_gap = _ancho(W.FloatText(value=0.0))
w_ver = W.Dropdown(options=[("Con aproximación (X entera)", "ent"), ("Sin aproximación (X continua)", "cont")],
                   value="ent", description="Ver solución:", style={"description_width": "110px"},
                   layout=W.Layout(width="420px"))
b_gen = W.Button(description="Paso 2 · Generar tablas", button_style="info", icon="table",
                 tooltip="Crear las tablas de datos según la estructura de arriba", layout=W.Layout(width="230px", height="40px"))
b_ej = W.Button(description="Cargar caso del artículo", icon="book", button_style="warning",
                tooltip="Rellenar todo con el ejemplo del paper (4 etapas, 12 fuentes, 4 trimestres)",
                layout=W.Layout(width="230px", height="40px"))
b_res = W.Button(description="Paso 4 · RESOLVER", button_style="success", icon="play",
                 tooltip="Calcular el plan óptimo", layout=W.Layout(width="230px", height="44px"))
b_xls = W.Button(description="Descargar Excel", icon="download", layout=W.Layout(width="190px", height="44px"))
msg = W.HTML("")
w_vista = W.HTML("")
caja_tablas = W.VBox([])
o_res, o_tab, o_plan, o_grafo, o_graf, o_util = (W.Output() for _ in range(6))
w_filtro = W.Dropdown(options=["Todas", "Solo valor ≠ 0", "Producción (X)", "Inventarios (I)", "Preparaciones (Y)"],
                      value="Solo valor ≠ 0", description="Mostrar:")
res_tabs = W.Tab(children=[o_res, W.VBox([w_filtro, o_tab]), o_plan, o_grafo, o_graf, o_util])
for k_tab, t_name in enumerate(["Resumen", "Tabla solución", "Plan por etapa", "Grafo", "Gráficas", "Utilización"]):
    res_tabs.set_title(k_tab, t_name)

def aviso(txt, color="#c0392b"):
    msg.value = (f"<div style='border-left:6px solid {color};background:{rgba(color, .13)};padding:8px 14px;"
                 f"border-radius:8px;margin:6px 0;max-width:1000px'><b style='color:{color}'>{txt}</b></div>")

def _etq(t, c):
    return (f"<span style='background:{rgba(c, .25)};border:1px solid {c};border-radius:14px;padding:2px 10px;"
            f"margin:2px;display:inline-block;font-weight:700'>{t}</span>")

def leer_estructura():
    prods = [s.strip() for s in w_prod.value.split(",") if s.strip()]
    try: J = [int(s) for s in w_J.value.replace(";", ",").split(",") if s.strip()]
    except ValueError: raise ValueError("'Fuentes por etapa' debe tener solo números separados por coma. Ejemplo: 3,2,3,4")
    T = int(w_T.value)
    if not prods: raise ValueError("Debe haber al menos un producto.")
    if not J or min(J) < 1: raise ValueError("Cada etapa necesita al menos 1 fuente. Ejemplo: 3,2,3,4")
    if T < 1: raise ValueError("Debe haber al menos 1 período.")
    return prods, J, T

def actualizar_vista(_=None):
    try: prods, J, T = leer_estructura()
    except Exception as e:
        w_vista.value = f"<div style='color:#c0392b'><b>{e}</b></div>"; return
    M = len(J)
    flujo = " <b>→</b> ".join(_etq(f"Etapa {i} · {n} fuente{'s' if n > 1 else ''}", COL_T["cap"]) for i, n in enumerate(J, 1))
    w_vista.value = (
        f"<div style='border:2px dashed #8886;border-radius:10px;padding:10px 14px;max-width:1000px'>"
        f"<b>Estructura resultante:</b><div style='margin-top:6px'>Productos: "
        + "".join(_etq(p, COL_T["dem"]) for p in prods) + "</div>"
        f"<div style='margin-top:6px'>Proceso: {flujo} <b>→</b> {_etq('Producto terminado', '#27ae60')}</div>"
        f"<div style='margin-top:6px'>Períodos: " + "".join(_etq(f"T{t}", COL_T["etp"]) for t in range(1, T + 1)) + "</div>"
        f"<div style='margin-top:6px;opacity:.85'>= {len(prods)} producto(s) · {M} etapa(s) · {sum(J)} fuente(s) en total · {T} período(s)</div></div>")

for _w in (w_prod, w_J, w_T): _w.observe(actualizar_vista, names="value")

def generar_tablas(_=None):
    try: prods, J, T = leer_estructura()
    except Exception as e: aviso(str(e)); return
    M = len(J); P = len(prods)
    src = [(i, j) for i in range(1, M + 1) for j in range(1, J[i - 1] + 1)]
    cT = [f"T{t}" for t in range(1, T + 1)]
    rows_pf = [f"{p} · Etapa {i} · Fuente {j}" for p in prods for (i, j) in src]
    g = {}
    g["dem"] = Grid(prods, ["Inv. inicial", "Inv. final", "Costo escasez"] + [f"Demanda T{t}" for t in range(1, T + 1)],
                    [0, 0, 10] + [10000] * T, "dem", ["unidades", "unidades", "$ por unidad"] + ["unidades"] * T)
    g["dias"] = Grid(["Días hábiles del período"], cT, [65] * T, "cap", ["días"] * T)
    g["fte"] = Grid([f"Etapa {i} · Fuente {j}" for i, j in src],
                    ["Nº máquinas", "Pérdida mantenim.", "Turnos", "Horas por turno"], [5, 12, 1, 8], "cap",
                    ["máquinas", "horas/mes por máquina", "turnos al día", "horas"], ancho=130)
    g["tasa"] = Grid(rows_pf, ["Tasa de producción", "Costo de preparación"], [25, 20000], "tasa",
                     ["unidades/hora por máquina", "$ por vez"], ancho=150)
    g["costo"] = Grid(rows_pf, cT, [20] * T, "costo", ["$ por unidad" ] * T)

    # Modificar pestaña de inventarios para incluir stock inicial y final por etapa
    g["etp"] = Grid([f"{p} · Etapa {i}" for p in prods for i in range(1, M + 1)],
                    ["Costo de mantener", "Consumo por unidad", "Inv. Inicial Etapa", "Inv. Final Etapa"], [1, 1, 0, 0], "etp",
                    ["$ por unidad", "coeficiente", "unidades", "unidades"], ancho=150)
    S["g"], S["est"] = g, (prods, J, T)
    H = lambda *a, **k: ayuda(*a, **k)
    tabs = W.Tab(children=[
        W.VBox([H("Tabla 1 · Demanda e inventarios", COL_T["dem"],
                  "Datos de producto terminado al empezar y terminar el horizonte.", [], ""), g["dem"].box]),
        W.VBox([H("Tabla 2 · Capacidad de las máquinas", COL_T["cap"], "", [], ""), g["dias"].box, W.HTML("<div style='height:14px'></div>"), g["fte"].box]),
        W.VBox([H("Tabla 3 · Velocidad y preparación", COL_T["tasa"], "", [], ""), g["tasa"].box]),
        W.VBox([H("Tabla 4 · Costo de producir", COL_T["costo"], "", [], ""), g["costo"].box]),
        W.VBox([H("Tabla 5 · Inventarios e Inventario Inicial/Final de Etapa", COL_T["etp"],
                  "Aquí debes ingresar el costo de mantener inventario y el inventario inicial/final para cada etapa.",
                  [("Costo de mantener", "Costo por almacenar cada unidad por período."),
                   ("Consumo por unidad", "Unidades consumidas de la etapa previa para generar una de la actual."),
                   ("Inv. Inicial Etapa", "Fija el inventario disponible al inicio de la etapa (ej. 100, 50, 150)."),
                   ("Inv. Final Etapa", "Fija el inventario remanente obligatorio al terminar el plan en cada etapa.")], ""), g["etp"].box])])
    for k_tab, t_name in enumerate(["1 · Demanda", "2 · Capacidad", "3 · Velocidad y setup", "4 · Costos", "5 · Inventarios"]):
        tabs.set_title(k_tab, t_name)
    caja_tablas.children = [W.HTML("<h3>Paso 3 · Editar los datos</h3>"), tabs]

def cargar_ejemplo(_=None):
    a = caso_articulo()
    w_prod.value, w_J.value, w_T.value = "Producto", ",".join(map(str, a["J"])), a["T"]
    w_meses.value, w_efic.value, w_falt.value = a["meses"], a["efic"], True
    generar_tablas()
    g = S["g"]; src = sorted(a["fte"])
    g["dem"].set([[a["inv_ini"], a["inv_fin"], a["c_falt"]] + a["demanda"]])
    g["dias"].set([a["dias"]])
    g["fte"].set([[a["fte"][k][0], a["fte"][k][1], 1, 8] for k in src])
    g["tasa"].set([[a["rate"][k], a["setup"][k]] for k in src])
    g["costo"].set([a["cu"][k] for k in src])
    # Rellenar con ceros iniciales y finales para el caso artículo por defecto
    g["etp"].set([[h, 1, 0, 0] for h in a["mantener"]])
    aviso("Caso del artículo cargado.", "#27ae60")

def leer_datos():
    if S["est"] is None: raise ValueError("Primero hay que pulsar 'Generar tablas'.")
    prods, J, T = S["est"]; g = S["g"]; M = len(J); P = len(prods)
    src = [(i, j) for i in range(1, M + 1) for j in range(1, J[i - 1] + 1)]
    dem, dias, fte = g["dem"].get(), g["dias"].get()[0], g["fte"].get()
    tasa, costo, etp = g["tasa"].get(), g["costo"].get(), g["etp"].get()

    fuente = {k: dict(N=fte[n][0], perd=fte[n][1], turnos=fte[n][2], horas=fte[n][3]) for n, k in enumerate(src)}

    inv_ini_etapas = {}
    inv_fin_etapas = {}

    for p in range(P):
        for i in range(1, M + 1):
            idx_etp = p * M + i - 1
            inv_ini_etapas[(p, i)] = etp[idx_etp][2]
            inv_fin_etapas[(p, i)] = etp[idx_etp][3]

    d = Datos(productos=prods, J=J, T=T, meses=w_meses.value, efic=w_efic.value, dias=list(dias), fuente=fuente,
              tasa={}, setup={}, costo={}, mantener={}, coef={},
              demanda={(p, t + 1): dem[p][3 + t] for p in range(P) for t in range(T)},
              inv_ini=list(dem[:, 0]), inv_fin=list(dem[:, 1]), c_falt=list(dem[:, 2]),
              inv_ini_etapas=inv_ini_etapas, inv_fin_etapas=inv_fin_etapas,
              faltantes=w_falt.value, enteras=True, t_limite=w_tl.value, gap=w_gap.value)

    for p in range(P):
        for n, (i, j) in enumerate(src):
            r = p * len(src) + n
            d.tasa[(p, i, j)], d.setup[(p, i, j)] = tasa[r][0], tasa[r][1]
            for t in range(T): d.costo[(p, i, j, t + 1)] = costo[r][t]
        for i in range(1, M + 1):
            d.mantener[(p, i)], d.coef[(p, i)] = etp[p * M + i - 1][0], etp[p * M + i - 1][1]
    return d

def pintar_tabla(_=None):
    R = S["R"]
    if not R: return
    t = R["tabla"]; f = w_filtro.value
    if f == "Solo valor ≠ 0": t = t[t["Valor solución"].abs() > 1e-9]
    elif f == "Producción (X)": t = t[t["Variable"].str.startswith("X")]
    elif f == "Inventarios (I)": t = t[t["Variable"].str.startswith("I")]
    elif f == "Preparaciones (Y)": t = t[t["Variable"].str.startswith("Y")]
    o_tab.clear_output()
    with o_tab:
        display(HTML(tabla_html(t, 480, color=PAL["gris"], fila_color=lambda r: TIPO_COLOR.get(r["Variable"][0], PAL["gris"]))))

w_filtro.observe(pintar_tabla, names="value")

def mostrar(R, d, etiqueta="", Rs=None):
    for o in (o_res, o_plan, o_grafo, o_graf, o_util): o.clear_output()
    c, tot = R["costos"], max(R["obj"], 1e-9)
    pct = lambda v: f"{100*v/tot:.1f}% del total"
    ok = R["estado"].startswith("Óptima")
    with o_res:
        display(HTML(f"<h4>Costo Total: $ {ES(R['obj'])}</h4><p>Estado: {R['estado']}</p>"))
    pintar_tabla()
    T = d.T
    with o_plan:
        for nombre in d.productos:
            df = R["plan"][nombre].reset_index().rename(columns={"index": "Fuente"})
            display(HTML(seccion(f"{nombre} · Producción", PAL["prod"], tabla_html(df, 380, 0, PAL["prod"], calor=True))))
            df = R["inv"][nombre].reset_index().rename(columns={"index": ""})
            display(HTML(seccion(f"{nombre} · Inventarios", PAL["mant"], tabla_html(df, 380, 0, PAL["mant"], calor=True))))

def _pintar_version(_=None):
    if S.get("mute") or not S.get("Rs"): return
    k = w_ver.value
    if k not in S["Rs"]: return
    S["R"] = S["Rs"][k]
    mostrar(S["R"], S["d"], "", S["Rs"])

w_ver.observe(_pintar_version, names="value")

def resolver_click(_=None):
    try: d = leer_datos()
    except Exception as e: aviso(str(e)); return
    modo = w_modo.value
    corridas = (["ent"] if modo[0] in "AE" else []) + (["cont"] if modo[0] in "AC" else [])
    aviso("Resolviendo…", "#2980b9")
    Rs = {}
    try:
        for k in corridas:
            Rs[k] = resolver(replace(d, enteras=(k == "ent")))
    except Exception as e:
        aviso(f"Error al resolver: {e}"); return
    S["Rs"], S["d"] = Rs, d
    k = "ent" if "ent" in Rs else "cont"
    S["mute"] = True; w_ver.value = k; S["mute"] = False
    S["R"] = Rs[k]
    mostrar(Rs[k], d, "", Rs)
    aviso("¡Optimización finalizada con éxito!", "#27ae60")

b_gen.on_click(generar_tablas); b_ej.on_click(cargar_ejemplo); b_res.on_click(resolver_click)

def _form(filas):
    items = []
    for t, w, h in filas:
        items += [W.HTML(f"<b>{t}</b>"), w, W.HTML(f"<span style='opacity:.85'>{h}</span>")]
    return W.GridBox(items, layout=W.Layout(grid_template_columns="170px 290px 1fr", grid_gap="8px 14px", align_items="center"))

ui = W.VBox([
    W.HTML("<h2>Planeación Agregada</h2>"),
    _form([
        ("Productos", w_prod, ""),
        ("Fuentes por etapa", w_J, ""),
        ("Nº de períodos", w_T, ""),
        ("Meses por período", w_meses, ""),
        ("Factor de eficiencia", w_efic, ""),
        ("Escasez", w_falt, ""),
        ("Resolver", w_modo, ""),
    ]),
    w_vista, b_gen, b_ej, caja_tablas, b_res, w_ver, res_tabs
])
display(ui)
actualizar_vista()
generar_tablas()

KeyError: 'costos'

KeyError: 'costos'